# Alt Yazi Cikarici - Google Colab

Bu defteri proje klasoruyle birlikte Google Drive'a yukleyin. `videolar/`, `src/`, `pyproject.toml` ve bu notebook ayni proje kokunde durmali.

Yukaridan asagiya calistirdiginizde Drive baglanir, proje bulunur, bagimliliklar kurulur ve `videolar` klasorundeki videolar icin Turkce veya Ingilizce dil algilamasina gore `.srt` dosyalari uretilir.

In [ ]:
# 1) Google Drive'i bagla ve proje klasorunu bul
from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount('/content/drive')

# Gerekirse bu satiri kendi Drive yolunuza gore degistirin.
PROJECT_DIR = Path('/content/drive/MyDrive/alt_yazi_cikarici')

def looks_like_project(path: Path) -> bool:
    return (
        (path / 'pyproject.toml').exists()
        and (path / 'src' / 'altyazi_cikarici').exists()
        and (path / 'videolar').exists()
    )

if not looks_like_project(PROJECT_DIR):
    candidates = []
    for base in [Path('/content/drive/MyDrive'), Path('/content/drive/Shareddrives')]:
        if base.exists():
            candidates.extend(p.parent for p in base.rglob('pyproject.toml') if looks_like_project(p.parent))
    if candidates:
        PROJECT_DIR = candidates[0]
    else:
        raise FileNotFoundError(
            'Proje klasoru bulunamadi. PROJECT_DIR degiskenini Drive icindeki proje kokune ayarlayin.'
        )

os.chdir(PROJECT_DIR)
SRC_DIR = PROJECT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f'Calisma dizini: {PROJECT_DIR}')
print(f'Python import yolu eklendi: {SRC_DIR}')
print(f'Videolar klasoru: {PROJECT_DIR / "videolar"}')

In [ ]:
# 2) Sistem ve Python bagimliliklarini kur
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!python -m pip install -q --upgrade pip
!python -m pip install -q -e .

In [ ]:
# 3) Calisma ayarlari
# Yereldeki varsayilan akisa en yakin ayarlar: model=large, naming-style=lesson-lab.
# Colab'da GPU actiysaniz DEVICE otomatik cuda olur; Runtime > Change runtime type > T4 GPU secilebilir.
import subprocess

SOURCE_DIR = PROJECT_DIR / 'videolar'
MODEL = 'large'
NAMING_STYLE = 'lesson-lab'
gpu_check = subprocess.run(['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
DEVICE = 'cuda' if gpu_check.returncode == 0 else 'cpu'

print('SOURCE_DIR =', SOURCE_DIR)
print('MODEL      =', MODEL)
print('DEVICE     =', DEVICE)
print('NAMING     =', NAMING_STYLE)

In [ ]:
# 4) Video klasorlerini kontrol et
from altyazi_cikarici.main import find_all_video_dirs

video_dirs = find_all_video_dirs(str(SOURCE_DIR))
print(f'Video iceren klasor sayisi: {len(video_dirs)}')
for path in video_dirs[:30]:
    print('-', path)
if len(video_dirs) > 30:
    print(f'... {len(video_dirs) - 30} klasor daha var')
if not video_dirs:
    raise FileNotFoundError(f'Video bulunamadi: {SOURCE_DIR}')

In [ ]:
# 5) Mevcut videolardan altyazi cikar
# Zaten var olan .srt dosyalari paket tarafindan atlanir.
import subprocess
import sys

cmd = [
    sys.executable,
    '-u',
    '-m', 'altyazi_cikarici.main',
    '-s', str(SOURCE_DIR),
    '--transcribe-only',
    '-m', MODEL,
    '-d', DEVICE,
    '--naming-style', NAMING_STYLE,
]
print('Komut:', ' '.join(cmd), flush=True)
print('Not: Model ilk kez indiriliyorsa ozellikle large modelde uzun sure sessiz kalabilir.', flush=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC_DIR) + os.pathsep + env.get('PYTHONPATH', '')
env['PYTHONUNBUFFERED'] = '1'
process = subprocess.Popen(
    cmd,
    stdin=None,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)
for line in process.stdout:
    print(line, end='', flush=True)
return_code = process.wait()
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, cmd)

In [ ]:
# 6) Uretilen SRT dosyalarini ozetle
srt_files = sorted(SOURCE_DIR.rglob('*.srt'))
print(f'Toplam SRT dosyasi: {len(srt_files)}')
for path in srt_files[-30:]:
    print(path.relative_to(PROJECT_DIR))

## Notlar

- Defter durdurulup tekrar calistirilirse mevcut `.srt` dosyalari yeniden uretilmez, atlanir.
- Sadece belirli bir alt klasoru islemek isterseniz `SOURCE_DIR` degerini oraya ayarlayin.
- Dil Turkce/Ingilizce olarak guvenle tespit edilemezse defter sizden `tr`, `en` veya `skip` secimi ister.
- Video indirme modunu calistirmak icin son komuttaki `SOURCE_DIR` yerine `dersler.json` verilip `--transcribe-only` kaldirilabilir.